# 数据预处理
:label:`sec_pandas`

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用`pandas`软件包。
像庞大的Python生态系统中的许多其他扩展包一样，`pandas`可以与张量兼容。
本节我们将简要介绍使用`pandas`预处理原始数据，并将原始数据转换为张量格式的步骤。
后面的章节将介绍更多的数据预处理技术。

## 读取数据集

举一个例子，我们首先(**创建一个人工数据集，并存储在CSV（逗号分隔值）文件**)
`../data/house_tiny.csv`中。
以其他格式存储的数据也可以通过类似的方式进行处理。
下面我们将数据集按行写入CSV文件中。


In [6]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True) #建一个文件夹
data_file = os.path.join('..', 'data', 'house_tiny.csv') #起个文件名
with open(data_file, 'w') as f: #a
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

要[**从创建的CSV文件中加载原始数据集**]，我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。其中每行描述了房间数量（“NumRooms”）、巷子类型（“Alley”）和房屋价格（“Price”）。


In [7]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
#!pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

      Chinese  Math.PE  Enlgish  total_score
90.0    100.0       60      110          300
95.0      NaN       50      100          245
99.0      NaN       66      101          266
NaN       NaN      100      100          200


In [5]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
#!pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

NameError: name 'data' is not defined

In [ ]:
%pip install pandas -i https://pypi.tuna.tsinghua.edu.cn/simple #安装pandas

## 处理缺失值

注意，“NaN”项代表缺失值。
[**为了处理缺失的数据，典型的方法包括*插值法*和*删除法*，**]
其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。
在(**这里，我们将考虑插值法**)。

通过位置索引`iloc`，我们将`data`分成`inputs`和`outputs`，
其中前者为`data`的前两列，而后者为`data`的最后一列。
对于`inputs`中缺少的数值，我们用同一列的均值替换“NaN”项。

In [15]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2] # iloc：pandas 里按“数字位置”切数据的工具。# 0:2 意思是取第 0 位和第 1 位，不包括第 2 位。
inputs = inputs.fillna(inputs.mean(numeric_only=True)) #inputs.mean 算平均分
print(inputs) 
# inputs（特征）：输入给模型的“题目”数据。
# outputs（标签）：我们希望模型预测的“标准答案”。
# 机器学习就是找规律：让模型学会根据 inputs 算出正确的 outputs。

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


[**对于`inputs`中的类别值或离散值，我们将“NaN”视为一个类别。**]
由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”，
`pandas`可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。
巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。
缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。


In [13]:
inputs = pd.get_dummies(inputs, dummy_na=True) # 新版 Pandas 里 get_dummies 默认输出 True/False。
                                               # # True 就是 1，False 就是 0，不影响后续转成 PyTorch 张量。
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


## 转换为张量格式

[**现在`inputs`和`outputs`中的所有条目都是数值类型，它们可以转换为张量格式。**]
当数据采用张量格式后，可以通过在 :numref:`sec_ndarray`中引入的那些张量函数来进一步操作。


In [14]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float)) # to_numpy(dtype=float)：把表格变数组，并强制变成浮点数（深度学习必须用 float）。 
                                               # torch.tensor()：把数组变成 PyTorch 张量。
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 小结

* `pandas`软件包是Python中常用的数据分析工具中，`pandas`可以与张量兼容。
* 用`pandas`处理缺失的数据时，我们可根据情况选择用插值法和删除法。

## 练习

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。


## 做练习

In [13]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'student_score.csv')
with open(data_file, 'w') as f:
    f.write('Chinese,Math,PE,Enlgish,total_score\n')  
    f.write('90,100,60,110,300\n')
    f.write('95,NA,50,100,245\n')
    f.write('99,NA,66,101,266\n')
    f.write('NA,NA,100,100,200\n')

In [14]:
data = pd.read_csv(data_file)
print(data) #用pd读取

   Chinese   Math   PE  Enlgish  total_score
0     90.0  100.0   60      110          300
1     95.0    NaN   50      100          245
2     99.0    NaN   66      101          266
3      NaN    NaN  100      100          200


In [19]:
inputs, outputs = data.iloc[:, 0:4], data.iloc[:, 4]
inputs = inputs.drop(columns=['Math'])
print(inputs)

   Chinese   PE  Enlgish
0     90.0   60      110
1     95.0   50      100
2     99.0   66      101
3      NaN  100      100


In [20]:
import torch

x = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))

x,y

(tensor([[ 90.,  60., 110.],
         [ 95.,  50., 100.],
         [ 99.,  66., 101.],
         [ nan, 100., 100.]], dtype=torch.float64),
 tensor([300., 245., 266., 200.], dtype=torch.float64))

# Day 4：2.2 数据预处理 学习笔记

## 知识点大白话总结（我的小抄）
- **os 模块**：用 `os.makedirs()` 建文件夹，用 `with open(..., 'w')` 造一个长满“缺失值”的假 CSV 文件，方便练习。
- **读取数据**：`pd.read_csv(data_file)`，把硬盘里的表格读进内存，变成 pandas 的 DataFrame。
- **处理缺失值（两大流派）**：
  - **填补法**：`data.fillna(data.mean())`，算个平均数把空缺填上。
  - **删除法**：`data.drop(columns=['列名'])`，哪里缺得太离谱，直接整列扔掉。
- **拆分题目和答案**：`inputs, outputs = data.iloc[:, 0:3], data.iloc[:, 3]`。大写 X 是特征，小写 y 是标签。
- **转换为张量**：`torch.tensor(inputs.to_numpy(dtype=float))`，强制转成浮点数，变成 PyTorch 能吃的张量。

## 今天遇到的坑与疑问
1. **`pd` 未定义报错**：`NameError: name 'pd' is not defined`。解决：忘了写 `import pandas as pd`，或者写在别的单元格里没运行。核心代码前后必须保持导入顺序。
2. **数据类型报错**：`TypeError: Cannot perform reduction 'mean' with string dtype`。解决：新版 pandas 里算平均值遇到文字列会报错，必须加上 `numeric_only=True`。
3. **True/False 问题**：`get_dummies` 默认输出布尔值（True/False），如果想变回 0 和 1，要在括号里加上 `dtype=int` 或者 `dtype=float`。
4. **iloc 切片越界**：如果数据集只有 3 列，写 `data.iloc[:, 0:3]` 会把答案也包含进去，导致报错。必须先数清楚列数，`左闭右开` 的规则要记牢。

## 今日感想（碎碎念）
- 今天最大的感受是：很多需要自己动手输入的代码，很多都可以实操了(ps:就是感觉调数据还不太熟练)
- 今天坚持完成了课后练习（自己造数据、删列、转张量），感觉非常有成就感，感觉距离“能自己写代码”又近了一步。

[Discussions](https://discuss.d2l.ai/t/1750)
